# Readme E-Codices (Handschriften)

Dieses Jupyter Notebook bereitet Datenobjekte und Metadaten aus dem E-codices-Bestand vor für die digitale Langzeitarchivierung (DLZA) der ZHB Luzern, basierend auf der GOCFL implementierung von Jürgen Enge, https://github.com/je4/gocfl .



## Basiskonfiguration

Stand Juli 2023:
20 Titel, davon 19 ZHB und 1 Korporation Luzern (Schilling Chronik)

Schnittstelle: 
Da die Daten auf E-Codices (unifr) sich stark unterscheiden von den Daten in Alma (z.B. engl. und lateinische Titel/Autorennamen, keine Verlinkung von DOI/Alma-ID), wurde eine Eingabedatei mit Signaturen, DOI und MMS ID erstellt, s.unten. Angesichts der kleinen Menge wurde diese Datei von Hand zusammengeführt. 


### Signature:

Für die E-Codices-Signaturen werden die DOI verwendet, welche sich aus den Original-Signaturen ableiten lassen. 
Bsp. doi:10.5076/e-codices-zhl-0034-4 / call number:  Msc.34.4 

Signature Beispiel:

    zhb_10_5076_e-codices-zhl-0034-4



### Config.py

Beispieldaten für ZHB E-codices. Anpassungen können in der config.py vorgenommen werden. 

    collection = 'ZHB E-Codices'
    collection_id = 'zhb_sosa_e-codices'
    ingest_workflow = 'zhb_ecodices'
    keywords = '[E-Codices, ZHB, Sondersammlung]'
    organisation = 'Zentral- und Hochschulbibliothek Luzern'
    organisation_id = 'zhb'
    sets = '[ecodices, zhb, sosa, lara]'
    signature = 'zhb_'

    
### Eingabedatei

Die Excel-Datei liegt im working directory. Sie kann relativ leicht in Alma exportiert werden. Die E-codices sind in folgendem Set in der RZS gelistet: 

    Se-codices_dlza_heka - Itemized 

Die Export-Datei wurde leicht überarbeitet. Nicht benötigte Spalten werden gelöscht, einige Datenmüssen gesplitted werden:

- Erstellungsdatum aus erster Spalte extrahieren, umbenennen zu Created
- Signatur aus Spalte Availability splitten, umbenennen zu Call number
- MMS_ID als Text erzwingen, indem man ein x anhängt
- Dateipfade händisch ergänzen. Gibt es mehrere Dateien pro ID, als Array erfassen (bsp. siehe Schillingchronik)
- DOI händisch ergänzen
- ARK händisch ergänzen (derzeit nur bei Schillingchronik vorhanden)

Da die Sammlung überschaubar ist, hält sich der zeitliche Aufwand dafür in Grenzen.


### Export

Für jeden einzelnen record wird eine info.json-Datei erstellt im Format signature.json im directory info.
Das ganze Set wird am Ende noch als json- und Excel-Datei exportiert ins directory files.

Zusätzlich wird ein Textfile mit den signatures.txt im fulldump-directory abgelegt. Sie wird für die Erstellung der Create-Befehle benötigt.

### Marcxml aus Alma (SRU)

Mit der alma_id werden die MARC-Daten via SRU aus Alma extrahiert und abgespeichert im directory metadata unter signature.xml


### TODO Datenobjekte abholen
Die E-Codices-Objekte liegen auf G:\ZHB-Sosa_Digital\digital unter folgenden Pfaden:

Msc\ecod_Msc...
P\ecod_P...
Romero (Signatur)\ecod_Romero...
S\ecod_...

Der aktuelle Dateipfad ist in der Excel-Datei in Spalte "filepath" abgelegt und wird auch in die Infojson unter 'additional' abgelegt. 


In [1]:
import json
import config
import pandas as pd
import requests
from datetime import datetime
import os
import shutil

# Read the Excel file into a pandas DataFrame

input_file = "e-codices.xlsx"
df = pd.read_excel(input_file)

# Convert the DataFrame to a list of dictionaries

completeSet = []

# other variables
today = datetime.today().strftime('%Y-%m-%d')
sigfile = "files/signatures.txt"

if os.path.exists('files'):    
    if os.path.exists(sigfile):
        os.remove(sigfile)
else:
    os.mkdir('files')

if os.path.exists('info'):
    shutil.rmtree('info')
    os.mkdir('info')
else:
    os.mkdir('info')
    
if os.path.exists('metadata'):
    shutil.rmtree('metadata')
    os.mkdir('metadata')
else:
    os.mkdir('metadata')

for _, row in df.iterrows():
    
    doi = row['DOI']
    mms_id = row['MMS ID'][:-1]
    print(mms_id)
    han_id = row['Record number']
    callno = row['Call number']
    doiurl = config.urldoi+doi
    almaurl = config.urlalma+mms_id

    
    infoSet = {
        'additional': row['Dateipfad'].replace('\\','/'),
        'address': config.address, 
        'collection': config.collection,
        'collection_id': config.collection_id ,
        'created': str(row['Created']), 
        'identifiers': [doi, mms_id, han_id, callno],
        'ingest_workflow': config.ingest_workflow, 
        'keywords': config.keywords, 
        'last_changed': today,
        'organisation' : config.organisation,
        'organisation_id' : config.organisation_id,
        'references' : [doiurl, almaurl],
        'signature': config.signature+doi.replace('.','_').replace('/','_'),
        'sets' : config.sets,
        'title' : row['Title'],
        'user' : config.user      
    }
    
    signature = infoSet['signature']
    #print(signature)
    completeSet.append(infoSet)
    
    # Write the infoSet to a JSON file
    info_json = json.dumps(infoSet, indent=4, ensure_ascii=False)
    
    infofile = f"info/{signature}.json"
    with open(infofile, "w") as outfile:
        outfile.write(info_json)
        print(f"info.json saved as {infofile}")
    
    # get MARCXML metadata via Alma SRU 
    sru_url = "https://slsp-rzs.alma.exlibrisgroup.com/view/sru/41SLSP_RZS"
    query = f"{sru_url}?version=1.2&operation=searchRetrieve&recordSchema=marcxml&query=rec.id={mms_id}"
    response = requests.get(query)
    if response.status_code != 200:
        raise Exception(f"SRU request failed with status code {response.status_code}")

    # Save the response content (MARCXML) to a file
    if os.path.exists('metadata/{signature}'):
        pass
    else:
        os.mkdir(f'metadata/{signature}')
        
    metafile = f"metadata/{signature}/{signature}.xml"
    with open(metafile, 'wb') as file:
        file.write(response.content)
        print(f"Record with MMS ID {mms_id} saved as {metafile}") 
        
    # TODO get dc Data via e-codices, OAI sets (unifR), or from Zentralgut  
    #https://www.e-codices.unifr.ch/oai/oai.php?verb=ListRecords&metadataPrefix=oai_dc&set=kol (1 titel)
    #https://www.e-codices.unifr.ch/oai/oai.php?verb=ListRecords&metadataPrefix=oai_dc&set=zhl (19 titel)
    
    # write signature to file
    with open(sigfile, 'a') as file:
        file.write(signature)
        file.write("\n")
        print(f"signatures appended to {sigfile}\n")


# Writing completeSet as json file

fulldump = json.dumps(completeSet, indent=4, ensure_ascii=False)
fulljsonfile = "files/ecodices_complete_set.json"
with open(fulljsonfile, "w") as outfile:
    outfile.write(fulldump)
    print(f"---\nAll JSON written to {fulljsonfile}")
    
# Writing completeSet as Excel file
fullexcelfile = "files/ecodices_complete_set.xlsx"
df_json = pd.read_json(fulljsonfile)
df_json.to_excel(fullexcelfile)
print(f"Saved E-Codices in excel file as {fullexcelfile}")


9914249443205505
info.json saved as info/zhb_10_5076_e-codices-zhl-0034-4.json
Record with MMS ID 9914249443205505 saved as metadata/zhb_10_5076_e-codices-zhl-0034-4/zhb_10_5076_e-codices-zhl-0034-4.xml
signatures appended to files/signatures.txt

9914249441205505
info.json saved as info/zhb_10_5076_e-codices-zhl-0040.json
Record with MMS ID 9914249441205505 saved as metadata/zhb_10_5076_e-codices-zhl-0040/zhb_10_5076_e-codices-zhl-0040.xml
signatures appended to files/signatures.txt

9914249440305505
info.json saved as info/zhb_10_5076_e-codices-zhl-0042.json
Record with MMS ID 9914249440305505 saved as metadata/zhb_10_5076_e-codices-zhl-0042/zhb_10_5076_e-codices-zhl-0042.xml
signatures appended to files/signatures.txt

9914249439505505
info.json saved as info/zhb_10_5076_e-codices-zhl-0045.json
Record with MMS ID 9914249439505505 saved as metadata/zhb_10_5076_e-codices-zhl-0045/zhb_10_5076_e-codices-zhl-0045.xml
signatures appended to files/signatures.txt

9914249439005505
info.json